# 🧱 05 - Column Operations: Create, Update, Rename, Drop, and Transform

This notebook is self-contained — it builds its own sample data.

## ✅ Learning Goals
- Create new columns from calculations, text, True/False tests, bands (`pd.cut()`), and `.str`/`.dt` methods
- Update only the rows that match a condition with `.loc[condition, 'column']`, including several rules in the right order
- Rename one or multiple columns, and standardize messy column names
- Drop columns (and rows) safely, without accidentally changing the original
- Apply a function across a column with `.apply()`, or across a row with `.apply(axis=1)`
- Choose between a whole-column operation, `.apply()`, and a loop

> **The core habit:** create columns with whole-column operations, update conditional rows with `.loc[]`, and loop only when the task genuinely needs one row at a time.

## 1. Setup and Sample Data

In [ ]:
import pandas as pd

data = {
    'first_name': ['Alice', 'Bob', 'Charlie', 'David'],
    'last_name': ['Smith', 'Jones', 'Brown', 'Wilson'],
    'math_score': [85, 90, 78, 92],
    'science_score': [88, 85, 82, 95],
}

df = pd.DataFrame(data)
df

## 2. Creating New Columns

In [ ]:
# Simple column arithmetic
df['average_score'] = (df['math_score'] + df['science_score']) / 2
df

In [ ]:
# Combine two text columns
df['full_name'] = df['first_name'] + ' ' + df['last_name']
df

Either way, pandas does the work for **every row at once**: `math column + science column → average column`.

### True/False Columns
A comparison makes a column of `True`/`False`, which is handy for flagging rows:

In [ ]:
df['passed_both'] = (df['math_score'] >= 80) & (df['science_score'] >= 80)
df[['first_name', 'math_score', 'science_score', 'passed_both']]

### Bands with `pd.cut()`
`pd.cut()` sorts numbers into labeled **bands** (bins). With `right=False`, each band includes its left edge but not its right, so `[80, 90)` means 80 up to (but not including) 90:

In [ ]:
df['band'] = pd.cut(
    df['average_score'],
    bins=[0, 80, 90, float('inf')],
    labels=['C', 'B', 'A'],
    right=False,
)
df[['first_name', 'average_score', 'band']]

### Columns from `.str` and `.dt` Methods
Text columns have `.str` methods (notebook 06 has more), and date columns have `.dt` methods. Both work on the whole column:

In [ ]:
df['initials'] = df['first_name'].str[0] + df['last_name'].str[0]
df['shout'] = df['last_name'].str.upper()
df[['first_name', 'last_name', 'initials', 'shout']]

In [ ]:
sessions = pd.DataFrame({'session_date': pd.to_datetime(['2026-01-05', '2026-02-14', '2026-02-28'])})
sessions['month'] = sessions['session_date'].dt.month
sessions['weekday'] = sessions['session_date'].dt.day_name()
sessions

### Updating Only Some Rows
To change a column **only where a condition is true**, use `.loc[]` with a mask for the rows and a column name:

```python
df.loc[condition, 'column_to_change'] = new_value
```

Here, everyone under 80 in math gets a 5-point curve:

In [ ]:
df['curved_math'] = df['math_score']                                        # start as a copy
df.loc[df['math_score'] < 80, 'curved_math'] = df['math_score'] + 5          # change only some rows
df[['first_name', 'math_score', 'curved_math']]

💡 Notice the first line. If you skip it, the rows that **don't** match the condition get `NaN` in the new column, because nothing was ever assigned to them.

### Several Rules, One Column: Order Matters
Start with a default, then apply each rule with its own `.loc[]`. When a row matches **more than one** rule, the **last** rule wins, so put the most specific rule last:

In [ ]:
df['award'] = 'None'
df.loc[df['average_score'] >= 85, 'award'] = 'Honor Roll'
df.loc[(df['math_score'] >= 90) & (df['science_score'] >= 90), 'award'] = 'High Honors'   # most specific, so last
df[['first_name', 'math_score', 'science_score', 'average_score', 'award']]

David qualifies for both awards. If the Honor Roll rule ran **second**, it would overwrite his High Honors.

### Several Columns at Once
One condition can update several columns in a single `.loc[]`. Give it a list of columns and a matching list of values:

In [ ]:
df['needs_tutor'] = False
df.loc[df['average_score'] < 81, ['award', 'needs_tutor']] = ['Needs Support', True]
df[['first_name', 'average_score', 'award', 'needs_tutor']]

⚠️ Always change values with **one** `.loc[]` step. `df[mask]['col'] = value` (chained assignment) quietly changes a temporary copy instead of `df`. Notebook 03 demonstrates why.

## 3. Renaming Columns

In [ ]:
# Rename specific columns: {old name: new name}
df = df.rename(columns={'math_score': 'Math', 'science_score': 'Science'})
df.columns.tolist()

💡 **Reassigning vs. `inplace=True`.** `df = df.rename(...)` builds a new DataFrame and points `df` at it. That's the style pandas' own docs recommend, and it lets you check the result before you overwrite anything. You'll also see `df.rename(columns={...}, inplace=True)`, which changes `df` directly. Both work, so be ready to read either.

In [ ]:
# Rename ALL columns at once
df.columns = [col.upper() for col in df.columns]
df

Replacing the **whole** `.columns` list works, but it's risky: the new list must have exactly the right **length** and **order**, or every name shifts onto the wrong column without any error. To rename a few columns, `.rename()` with a dictionary is safer, especially in wide DataFrames.

### Standardizing Messy Column Names
Files often arrive with names like `' First Name'` or `'Math Score '`. Chaining `.str` methods on `.columns` cleans them all at once into tidy `snake_case`:

In [ ]:
messy = pd.DataFrame(columns=[' First Name', 'Math Score ', 'SCIENCE score'])
print('before:', messy.columns.tolist())

messy.columns = messy.columns.str.strip().str.lower().str.replace(' ', '_')
print('after: ', messy.columns.tolist())

## 4. Dropping Columns

In [ ]:
df = df.drop(columns=['FULL_NAME'])
df.columns.tolist()

`.drop()` returns a **new** DataFrame. If you don't assign it back, the original is untouched, which is useful for making a trimmed copy for a report:

In [ ]:
report = df.drop(columns=['SHOUT', 'INITIALS', 'NEEDS_TUTOR'])
print('report:', report.columns.tolist())
print('df still has', len(df.columns), 'columns')

### When a Column Might Not Exist
Dropping a column that isn't there raises a `KeyError`. Files from different sources don't always have the same columns, so there are two ways to handle it:

In [ ]:
to_remove = ['SHOUT', 'INTERNAL_CODE']           # INTERNAL_CODE doesn't exist

# Option 1: drop only the ones that are actually there
existing = [col for col in to_remove if col in df.columns]
print('dropping:', existing)

# Option 2: tell .drop() to skip missing ones
trimmed = df.drop(columns=to_remove, errors='ignore')
print('SHOUT' in trimmed.columns)

Use `errors='ignore'` deliberately. It prevents the error, but it can also **hide** a typo or a column you expected to be there.

### Columns vs. Rows
`.drop()` removes rows too. The keyword says which:

In [ ]:
print(df.drop(columns=['SHOUT']).shape)    # columns= → remove columns
print(df.drop(index=[0, 1]).shape)          # index=   → remove rows labeled 0 and 1

## 5. Applying a Function Across a Column
Remember writing your own small functions in Module 03b (the Containers → Functions module)? `.apply()` is where that pays off — you write a normal function, and Pandas runs it on every value in a column.

In [ ]:
def grade(score):
    if score >= 90:
        return 'A'
    elif score >= 80:
        return 'B'
    else:
        return 'C'

df['GRADE'] = df['AVERAGE_SCORE'].apply(grade)
df

For a one-line rule, a **`lambda`** (a small, unnamed function) saves writing a full `def`. Use a named function when the rule has several steps or you'll reuse it:

In [ ]:
df['IS_STRONG'] = df['AVERAGE_SCORE'].apply(lambda score: score >= 88)
df[['FIRST_NAME', 'AVERAGE_SCORE', 'IS_STRONG']]

### `.apply()` vs. a Whole-Column Operation
`.apply()` runs your function once per value, so it's still a loop under the hood. **Reach for a whole-column operation first**, and use `.apply()` when none exists:

| Instead of... | Write |
|---|---|
| `df['x'].apply(lambda v: v * 1.08)` | `df['x'] * 1.08` |
| `df['name'].apply(len)` | `df['name'].str.len()` |
| `df['score'].apply(lambda v: v >= 88)` | `df['score'] >= 88` |
| `df['score'].apply(grade)` (bands) | `pd.cut(df['score'], ...)` |

In fact, the `BAND` column made with `pd.cut()` in section 2 gives exactly the same letters as `grade()`:

In [ ]:
(df['BAND'].astype(str) == df['GRADE']).all()

### Across a Whole Row: `.apply(axis=1)`
When a rule needs **several columns**, `.apply(axis=1)` passes your function one **row** at a time, and you look up values by column name:

In [ ]:
def status(row):
    if row['MATH'] >= 90 and row['SCIENCE'] >= 90:
        return 'Excellent'
    elif row['MATH'] < 80 or row['SCIENCE'] < 80:
        return 'Review'
    return 'Solid'

df['STATUS'] = df.apply(status, axis=1)
df[['FIRST_NAME', 'MATH', 'SCIENCE', 'STATUS']]

That's convenient, but it's the slowest kind of `.apply()`. The same rule written as a default plus `.loc[]` rules is fully vectorized. Again, the most specific rule goes **last**:

In [ ]:
df['STATUS_FAST'] = 'Solid'
df.loc[(df['MATH'] < 80) | (df['SCIENCE'] < 80), 'STATUS_FAST'] = 'Review'
df.loc[(df['MATH'] >= 90) & (df['SCIENCE'] >= 90), 'STATUS_FAST'] = 'Excellent'

(df['STATUS'] == df['STATUS_FAST']).all()

## 6. Looping Over a DataFrame (and Why You Usually Shouldn't)
Sometimes you really do need to go row by row. Here are the ways to loop, from most to least useful:

In [ ]:
for column in df.columns:          # column names
    print(column, end='  ')
print()

for score in df['AVERAGE_SCORE']:  # the values of one column (a Series)
    print(score, end='  ')
print()

for row in df.itertuples():        # one row at a time, as a named tuple
    print(row.FIRST_NAME, row.AVERAGE_SCORE)

You'll also see `.iterrows()`, which gives `(index, row)` pairs. It works, but it's slower than `.itertuples()`, and it can quietly change data types. Prefer `.itertuples()` when you need a loop.

### The Vectorized Way Is Usually Better
For a calculation across a whole column, **don't loop at all**. A whole-column operation (vectorization, from Foundations notebook 08) is shorter, clearer, and much faster:

In [ ]:
import numpy as np

np.random.seed(3603)
stock = pd.DataFrame({
    'price': np.random.uniform(1, 50, size=10_000).round(2),
    'qty':   np.random.randint(0, 100, size=10_000),
})

# ❌ Row by row
def value_with_loop(frame):
    values = []
    for _, row in frame.iterrows():
        values.append(row['price'] * row['qty'])
    return values

# ✅ Whole columns at once
def value_vectorized(frame):
    return frame['price'] * frame['qty']

print('Same result?', np.allclose(value_with_loop(stock), value_vectorized(stock)))
t_loop = %timeit -o -q -n 1 -r 3 value_with_loop(stock)
t_vec = %timeit -o -q value_vectorized(stock)
print(f'Vectorized is about {t_loop.average / t_vec.average:,.0f}x faster')

### Choosing the Right Tool

```text
One rule for every row                  → a whole-column operation
Custom logic, one column                → .apply(func)
Custom logic using several columns      → several .loc[] rules, or .apply(func, axis=1)
Something to DO for each row            → a loop with .itertuples()
```

A loop is the right tool when each row triggers an **action**, not a calculation: writing a separate file per student, sending a customized message, or calling a website for each row. For calculating a column, it almost never is.

## 🏋️ Practice

### Practice 1 — Column from a Length (easy)
Add a column `'NAME_LENGTH'` containing the number of characters in each `FIRST_NAME`.

In [ ]:
# Your code here


### Practice 2 — Curved Scores (medium)
Create a column `'MATH_PLUS_5'` that's the `MATH` score plus a 5-point bonus, capped at 100 (hint: `.apply()` with a small function, or `.clip(upper=100)`).

In [ ]:
# Your code here


### Practice 3 — Apply with a Lambda (harder)
Redo Practice 2, but using a `lambda` instead of `.apply(grade)`-style named function: `df['MATH'].apply(lambda x: ...)`. Compare — which version do you find more readable?

In [ ]:
# Your code here


### Practice 4 — Loop vs. Vectorized (harder)
Load `students.csv`. Compute a `total_score` column (`math_score + science_score`) **twice**: once with an `.itertuples()` loop that builds a list, and once with a single whole-column expression. Check that the two results match, then explain in a comment which version is clearer, and why.

In [ ]:
students = pd.read_csv('students.csv')
# Your code here


### Practice 5 — Flags and Bands (medium)
Load `students.csv`. Add:
1. A True/False column `strong_science` for `science_score` of 90 or more.
2. A `math_band` column using `pd.cut()` with the bands `'Low'` (below 70), `'Mid'` (70 up to 85), and `'High'` (85 and up).

Then show how many students are in each `math_band` with `value_counts()`.

In [ ]:
students = pd.read_csv('students.csv')
# Your code here


## ✏️ Your Turn
Load `students.csv` (in this folder). Then:
1. Create a `'total_score'` column by summing `math_score` and `science_score`.
2. Rename `math_score` and `science_score` to `Math` and `Science`.
3. Drop any column that contains only `NaN` values (there may not be one — that's fine, just write code that would handle it if there were).

In [ ]:
students = pd.read_csv('students.csv')
# Your turn here


## 🔥 Challenge (Optional) — A Product Report
Here's a small product catalog. Using this notebook's tools, and **without loops**, turn it into a report:
1. Add `inventory_value` (`price * stock`) and a True/False `low_stock` column (stock below 15).
2. Add `discount_rate`: `0` by default, `0.10` for the Office department, and `0.15` for low-stock products. A low-stock Office product gets `0.15`, so think about the **order** of your rules.
3. Add `final_price` = `price * (1 - discount_rate)`, rounded to 2 decimal places.
4. Add a `price_category` with `pd.cut()`: Budget (under \$10), Standard (\$10 up to \$30), Premium (\$30 and up).
5. Build a `report` that drops `supplier_note` **without** changing `products`, and renames `stock` to `units_in_stock`.
6. Explain in a comment why `products.loc[condition, 'discount_rate'] = 0.10` is better than `products[condition]['discount_rate'] = 0.10`.

In [ ]:
products = pd.DataFrame({
    'product_id':    [101, 102, 103, 104, 105],
    'product_name':  ['Notebook', 'Backpack', 'Pen', 'Desk Lamp', 'Water Bottle'],
    'department':    ['Office', 'Travel', 'Office', 'Home', 'Travel'],
    'price':         [4.99, 39.99, 1.50, 24.99, 12.50],
    'stock':         [32, 14, 120, 8, 25],
    'supplier_note': ['standard', 'seasonal', 'clearance', 'standard', 'seasonal'],
})
# Your code here


---
### 📝 Summary
| Action              | Method                                 |
|---------------------|------------------------------------------|
| Create column       | `df['new'] = ...` (arithmetic, text, comparisons) |
| Put numbers in bands | `pd.cut(df['col'], bins=[...], labels=[...])` |
| Text / date parts   | `df['col'].str...`, `df['col'].dt...`   |
| Update some rows    | `df.loc[condition, 'col'] = value` (most specific rule last) |
| Update several columns | `df.loc[condition, ['a', 'b']] = [x, y]` |
| Rename column(s)    | `df = df.rename(columns={old: new})`    |
| Tidy all names      | `df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')` |
| Drop column(s)      | `df.drop(columns=[...])`, add `errors='ignore'` with care |
| Drop row(s)         | `df.drop(index=[...])`                  |
| Apply function      | `df['col'].apply(func)`, or `df.apply(func, axis=1)` for rows |
| Loop over rows      | `for row in df.itertuples():` (only when you must) |